In [2]:
import torch

device = torch.device("cuda:5") if torch.cuda.is_available() else "cpu"

In [3]:
import sys
import os

PROJECT_ROOT = "/tmp2/maitanha/vgu/cll_vlm"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PYTHONPATH updated")

import torch
from torch.utils.data import DataLoader
import pandas as pd
from tqdm import tqdm
from PIL import Image

from cll_vlm.dataset.cifar10 import CIFAR10Dataset
from cll_vlm.dataset.cifar20 import CIFAR20Dataset, CIFAR100Dataset

from cll_vlm.models import LLaVAClassifier, QWENClassifier

PYTHONPATH updated


/tmp2/maitanha/vgu/cll_vlm/venv_cll_qwen/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp2/maitanha/vgu/cll_vlm/cll_vlm/vlm/CLIP


In [4]:
batch_size = 256         
num_workers = 4              # số luồng đọc dữ liệu

dataset = CIFAR100Dataset(
    root="/home/maitanha/cll_vlm/cll_vlm/data/cifar100",
    train=True
)
fine_classes_raw = dataset.get_fine_classes()
fine_classes = [
    CIFAR100Dataset.preprocess_label(lbl)
    for lbl in fine_classes_raw
]
coarse_classes = dataset.get_coarse_classes()

print("Fine Labels:", fine_classes)
print("Coarse Labels:", coarse_classes)
print(f"Total samples: {len(dataset)}")

def collate_fn(batch):
    imgs, targets = zip(*batch)
    return list(imgs), torch.tensor(targets)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)

Fine Labels: ['apple', 'aquarium fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak tree', 'orange', 'orchid', 'otter', 'palm tree', 'pear', 'pickup truck', 'pine tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', '

In [3]:
import yaml

def load_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file '{config_path}' not found.")
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

config_path = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm/config/config.yaml"

config = load_config(config_path)
root_dir = config["workspace"]
data_cfg = config["data"]

# LLaVA Label Description

In [4]:
# === Cấu hình ===
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "qwen"
dataset_name = "cifar100"
save_dir = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm/ol_cll_logs"
save_path = os.path.join(save_dir, f"{model_name}_{dataset_name}_label_description.json")
os.makedirs(save_dir, exist_ok=True)

if model_name == "llava":
    model_path = config["models"][model_name]["model_url"]
    model = LLaVAClassifier(model_path=model_path)
elif model_name == "qwen" or model_name == "qwen3_2b" or model_name == "qwen3_8b":
    model_path = config["models"][model_name]["model_url"]
    model = QWENClassifier(model_path=model_path)
else:
    raise ValueError(f"Unsupported model '{model_name}'.")


# === PROMPT ===
system_content_general = "You are a helpful assistant who can describe any object in the world based on how it looks or where it is typically found."

prompt_general_visual = "What are the distinguishable characteristics that can be used to differentiate a {label} from other objects based on just a photo? \
                        Produce an exhaustive list of visual attributes, shape, color, structure, material, and contextual or habitat cues that help identify it visually. \
                        Each description should be a single sentence starting with 'An object that...'. \
                        The object name should not appear in the description. \
                        Structure your response as a list of single sentences."
                        
prompt_general_location = "In what environments or contexts is a {label} typically found or used? \
                        Produce a list of typical locations, habitats, or usage settings where one might expect to see it. \
                        Each sentence should begin with 'An object that is found in...' or 'An object typically used in...'. \
                        Do not mention the name of the object in any sentence. \
                        Structure your response as a list of single sentences."

def query_descriptions(label: str, model, system_prompt, visual_prompt, location_prompt):
    # Visual
    visual_text = visual_prompt.format(label=label)
    visual_response = model.generate_text(visual_text, system_content=system_prompt)
    visual_lines = [line.strip("-• ").strip() for line in visual_response.strip().split("\n") if line.strip()]
    visual_lines = [line for line in visual_lines if line.lower().startswith("an object")]

    # Context
    location_text = location_prompt.format(label=label)
    location_response = model.generate_text(location_text, system_content=system_prompt)
    location_lines = [line.strip("-• ").strip() for line in location_response.strip().split("\n") if line.strip()]
    location_lines = [line for line in location_lines if line.lower().startswith("an object")]

    return {
        "visual": visual_lines,
        "context": location_lines
    }

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


 [DEBUG] Model path: Qwen/Qwen2.5-VL-7B-Instruct


Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 68.97it/s]


In [5]:
import json

label_descriptions = {}
for label in tqdm(fine_classes, desc="Generating descriptions"):
    try:
        desc = query_descriptions(
            label=label,
            model=model,
            system_prompt=system_content_general,
            visual_prompt=prompt_general_visual,
            location_prompt=prompt_general_location
        )
    except Exception as e:
        desc = {"visual": [f"Error: {str(e)}"], "context": []}
    label_descriptions[label] = desc

# === Lưu kết quả ra JSON ===
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(label_descriptions, f, indent=2, ensure_ascii=False)

print(f"Saved label descriptions to {save_path}")

Generating descriptions: 100%|██████████| 100/100 [20:38<00:00, 12.39s/it]

Saved label descriptions to /tmp2/maitanha/vgu/cll_vlm/cll_vlm/ol_cll_logs/qwen_cifar100_label_description.json
